# Catch product-content mistakes before publication with the Decisions API

A catalog team receives product photos and campaign creatives from several suppliers. A file can look polished while showing the wrong product, a missing detail, or unreadable copy. Asking one model question, “Is this good?”, hides which requirement failed and whether the evidence was sufficient.

Build a review queue that checks each requirement separately. Combine a catalog record and approved reference image with candidate photos and campaign assets, then suggest an editor's next step. The example uses the [Decisions API](https://developers.openai.com/api/docs/guides/decisions), which returns typed assessments from fixed choices.

![Fictional campaign asset for a navy daypack](images/backpack_campaign.png)

You will define observable checks, preserve missing evidence as its own outcome, and compare the resulting review queue against local labels. The products and images are fictional.

## 1. Choose checks an editor can verify

| Check | Evidence | What a match means |
| --- | --- | --- |
| Listing color | Catalog JSON + listing photo | The bag's visible color agrees with the record. |
| Listing detail | Catalog JSON + listing photo | The required front pocket or patch is visible. |
| Campaign product | Approved reference + campaign image | Product color and front configuration are retained. |
| Campaign disclosure | Required copy + campaign image | The specified disclosure is visibly present. |

Each question returns `match`, `mismatch`, or `insufficient_evidence`. A rear view cannot prove a front patch is missing. Conversely, a complete, readable campaign without its required disclosure is a mismatch. The supplied photo-view metadata records how the fixture was photographed; the model does not infer the view from an expected label.

**Flow:** catalog + approved reference → four Decisions questions with candidate images → per-check assessments → deterministic review queue → local label comparison.

The model assesses visual evidence. Python chooses the workflow state. No content is published automatically.

## 2. Set up and inspect the evidence

Use Python 3.11 or later. From the repository root, install the example's dependencies in a virtual environment:

```bash
python3 -m venv .venv
.venv/bin/python -m pip install -r examples/decisions/product_content_review/requirements-dev.txt
```

Select this environment as the notebook kernel and open the notebook from its own directory. For a Jupyter server started elsewhere, change the kernel's working directory to `examples/decisions/product_content_review` first.

**Run All makes eight paid API requests, each with four questions.** Have a project API key with access to `gpt-6-luna`. The next cell uses `OPENAI_API_KEY`, or prompts for the key without displaying or saving it. Requests are spaced 5 seconds apart to reduce rate pressure. SDK retries and redirects are disabled. Costs depend on image sizes and token usage; see [API pricing](https://developers.openai.com/api/docs/pricing). The CLI's default run makes one request if you want a smaller first check.

In [ ]:
import getpass
import json
import os
import sys
import time
from pathlib import Path

from IPython.display import HTML, Image, JSON, display

sys.path.insert(0, str(Path.cwd()))

from evaluate import summarize
from report import write_report
from review import (ROOT, REQUEST_DELAY_SECONDS, build_request, create_client, load_cases,
                    record_response, request_preview, review_case)

data = load_cases()
cases = data["cases"]
output = ROOT / "results" / "notebook"
output.mkdir(parents=True, exist_ok=True)
for name in ["results.json", "report.html", "metrics.json", "executed.ipynb"]:
    (output / name).unlink(missing_ok=True)
key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key (hidden): ")
client = create_client(key)
del key
print(f"Ready: {len(cases)} paid requests, four checks each.")

In [ ]:
# Inspect the first candidate and its trusted catalog before calling the API.
case = cases[0]
product = data["products"][case["product_id"]]
display(JSON(product["catalog"]))
for label, image in [("Approved reference", product["reference_image"]),
                     ("Listing photo", case["listing_image"]),
                     ("Campaign asset", case["campaign_image"])]:
    print(label)
    display(Image(filename=str(ROOT / image), width=440))

## 3. Share the reference, attach each candidate to its question

The shared `input` contains a user message with `input_json` and the approved `input_image`. Each question supplies its own `instructions` content parts: the criterion as `input_text`, followed by its candidate `input_image`.

This lets listing checks and campaign checks use different images against the same record. The expected answers remain in `labels.json`, which is never read by the request builder. Case titles are also excluded from model input.

The next cell prints the request fields. The display replaces base64 image bytes with a marker; the actual request contains the images.

In [ ]:
payload = build_request(case, product)
display(JSON(request_preview(payload)))

## 4. Make one request and inspect the actual response

The API call itself is `client.decisions.create(**payload)`. Inspect each named answer's `choice`, `probabilities`, and `confidence`. Confidence is a model signal, not proof that a choice is correct, and it need not equal the selected choice's probability.

The parser associates answers by name, validates them against the choices in the request, and treats missing, duplicate, malformed, or refused answers as unusable. Other usable answers remain available.

The following answer was captured from the live `case_01` run on November 16, 2026. It is an observed example, not a guarantee for future requests:

```json
{
  "choice": "match",
  "confidence": 1.0,
  "name": "listing_color",
  "probabilities": [
    {
      "probability": 1.0,
      "value": "match"
    },
    {
      "probability": 0.0,
      "value": "mismatch"
    },
    {
      "probability": 0.0,
      "value": "insufficient_evidence"
    }
  ],
  "type": "choice"
}
```

Executing the next cell displays the complete response from your live request.

In [ ]:
# One actual Decisions request, shared with the CLI's implementation.
started = time.perf_counter()
response = client.decisions.create(**payload)
row = record_response(case, product, payload, response.model_dump(mode="json"),
                      round((time.perf_counter() - started) * 1000, 1))
records = [row]
display(JSON(row["response"]))
display(JSON({"checks": row["answers"], "next_step": row["next_step"]}))

## 5. Apply a small, explicit review policy

| Observed answers | Suggested next step |
| --- | --- |
| Any mismatch | `review_mismatch`: an editor inspects the flagged requirement. |
| No mismatch, but uncertainty or an unusable answer | `request_evidence`: obtain a better photo or manual assessment. |
| All checks match | `ready_for_editor_review`: proceed to an editor, with no automatic publication. |

This policy does not assume that a high-confidence match is safe to publish. If mismatch and uncertainty coexist, the row goes to mismatch review, and its unresolved checks remain visible.

Run the seven remaining cases. They cover a wrong product, a closed package, a rear view, and missing disclosure copy. Each case prints its actual request and response.

In [ ]:
# Save partial progress so a failed later request does not erase completed evidence.
def save_results():
    (output / "results.json").write_text(json.dumps(records, indent=2) + "\n")
    write_report(records, output / "report.html")

save_results()
try:
    for number, case in enumerate(cases[1:], 2):
        print(f"Waiting {REQUEST_DELAY_SECONDS}s before the next request...")
        time.sleep(REQUEST_DELAY_SECONDS)
        print(f"Example {number}/{len(cases)}: {case['title']}")
        row = review_case(client, case, data["products"][case["product_id"]])
        records.append(row)
        save_results()
        print("REQUEST (image bytes omitted)")
        display(JSON(row["request"]))
        print("ACTUAL API RESPONSE")
        display(JSON(row["response"]))
        print("SUGGESTED NEXT STEP:", row["next_step"])
finally:
    client.close()

## 6. Measure what the review queue misses

Now load the author-reviewed labels locally. Count missed mismatch flags, false mismatch flags, uncertainty incorrectly marked as a match, and cases suggested as editor-ready despite a labeled mismatch. Report counts and denominators, rather than presenting eight fixtures as production accuracy.

These are grouped examples, not independent training and held-out test sets. Images are deliberately reused to isolate content changes. The labels and prompts are illustrative and the sample is too small to establish calibration or generalization. A disagreement is something to inspect, not automatically relabel to make the model pass.

In [ ]:
labels = json.loads((ROOT / "labels.json").read_text())
metrics = summarize(records, labels)
(output / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
display(JSON(metrics))
print(f"Completed {len(records)} live requests.")
print(f"Open the side-by-side evidence report: {output / 'report.html'}")

## 7. Adapt the workflow to your catalog

1. Choose a few observable requirements. Do not ask an image to establish fabric composition, regulatory compliance, dimensions without a scale, or anything else it cannot show.
2. Replace the fictional catalog and references in `cases.json`. Keep references approved and capture metadata trustworthy; check schema-required fields deterministically before calling a model.
3. Update the question criteria and descriptions in `build_request`. Preserve an explicit insufficient-evidence outcome. Keep gold labels outside the request.
4. Label representative photos with a domain reviewer. Separate development and test products, suppliers, and image sources before tuning prompts or confidence thresholds.
5. Measure missed defects, false flags, manual workload, and performance on unclear photos separately. If you introduce a confidence threshold, select it on development data using the cost of those errors, then evaluate the frozen policy on the untouched test set.

For an application, store the evidence and assessments with the content version, give editors an override and correction path, and re-run checks when assets change. This recipe ends at a saved review report rather than a publishing integration.

### Files and provenance

`review.py` contains the request builder, SDK client and answer handling. Both this notebook and `run_review.py` use it. `evaluate.py` compares saved answers with local labels; `report.py` renders the evidence report. See [README.md](README.md) for both run paths and contributor checks.

The six original images were generated with OpenAI image generation for fictional products. No retailer photos, private customer data, or real brand guidelines are included. Image hashes and provenance are in [images/provenance.json](images/provenance.json). The sample disclosure is a demonstration criterion, not legal guidance.